In [1]:
!pip install plotly


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [502]:
!pip install plotly pandas


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [777]:
!pip install pycountry

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ------------------ --------------------- 3.7/8.0 MB 31.0 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 27.6 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [848]:
!pip install patsy 


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [850]:
!pip install Pylance

   ---------------------------------------- 0.0/74.8 MB ? eta -:--:--
   --- ------------------------------------ 6.0/74.8 MB 33.7 MB/s eta 0:00:03
   ----- ---------------------------------- 10.0/74.8 MB 25.9 MB/s eta 0:00:03
   ------ --------------------------------- 11.5/74.8 MB 19.0 MB/s eta 0:00:04
   ------- -------------------------------- 14.7/74.8 MB 17.8 MB/s eta 0:00:04
   --------- ------------------------------ 17.3/74.8 MB 16.8 MB/s eta 0:00:04
   ---------- ----------------------------- 20.4/74.8 MB 16.6 MB/s eta 0:00:04
   ------------- -------------------------- 26.0/74.8 MB 17.9 MB/s eta 0:00:03
   ---------------- ----------------------- 30.1/74.8 MB 18.2 MB/s eta 0:00:03
   ------------------ --------------------- 35.4/74.8 MB 19.1 MB/s eta 0:00:03
   ---------------------- ----------------- 41.2/74.8 MB 19.7 MB/s eta 0:00:02
   ------------------------- -------------- 46.9/74.8 MB 20.5 MB/s eta 0:00:02
   ---------------------------- ----------- 52.4/74.8 MB 21.0 


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Agriculture R&D Study - Data Collection

**Yelizaveta Kabanova - May/June 2026**

The main **objective** of this study is to investigate whether the effects of agricultural R&D investments differ across groups of structurally similar countries and whether these effects emerge over different time horizons. The study proposes a cluster-based dynamic response analysis framework that allows to identify different investment response profiles by estimating cluster-specific dynamic production responses to investment over time. 
Сountries are grouped into clusters based on economic, demographic, and environmental characteristics in order to identify cluster-specific investment horizons and characterise distinct response profiles. 

**The research hypothesis are as follows: **
- H1: Countries can be classified into distinct clusters based on economic, demographic and environmental characteristics.
- H2: The magnitude and timing of agricultural R&D effects differ across country clusters.
- H3: The causal effect of agricultural R&D investments is heterogeneous across country clusters.

**The objectives of this study are to:**
- construct a cross-country dataset by combining agricultural, economic, demographic, environmental, and R&D indicators;
- identify groups of structurally similar countries using clustering techniques;
- estimate the lagged effects of agricultural R&D investments on agricultural production within each cluster. 
- compare investment horizons across country clusters. 
- formulate cluster-specific structural response profiles based on observed dynamic effects.


In [1]:
# import os
import numpy as np
import pandas as pd
import pathlib 
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import pycountry
# from patsy import dmatrix
import plotly.express as px
from scipy.stats import zscore
# from scipy.stats.mstats import winsorize
# import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score # mean_squared_error, mean_absolute_error, r2_score, 
from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings
from pandas.errors import SettingWithCopyWarning
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=(SettingWithCopyWarning))

## Data Load

In [2]:
def read_csv(path):
    return pd.read_csv(path)

def print_unique(df, name='None'):
    print(f'\n-- {name} --\n{df["Country"].unique()}')

In [3]:
DIR = pathlib.Path(r"C:\Users\eliza\Portfolio\agricultural_investments")
DATA_DIR = pathlib.Path(r"C:\Users\eliza\Portfolio\agricultural_investments\data")

RD = read_csv(DATA_DIR / 'Agricultural R&D Investment.csv')   
PROD = read_csv(DATA_DIR / 'Value of Agricultural Production.csv')
GDP = read_csv(DATA_DIR / 'GDP.csv')
POP = read_csv(DATA_DIR / 'Population Total.csv')
POP_RUR = read_csv(DATA_DIR / 'Rural Population.csv')
RUR_LAND = read_csv(DATA_DIR / 'Rural Land.csv')
TEMP = read_csv(DATA_DIR / 'Average Temperature.csv')

sns.set_style("darkgrid")
sns.set_theme(palette="muted")

## Data Collection


**Variables:**
- Agricultural R&D investments
- Value of Agricultural Production
- GDP per capita (current US$) 
- Total population
- % of total population
- % of total land area
- Annual mean temperature

### Datasets

In [4]:
datasets = [
    (RD, 'Agricultural R&D Investment'),
    (PROD, 'Value of Agricultural Production'),
    (GDP, 'GDP'),
    (POP, 'Population Total'),
    (POP_RUR, 'Rural Population'),
    (RUR_LAND, 'Rural Land'),
    (TEMP, 'Average Temperature')
]

for df, name in datasets:
    print_unique(df, name)


-- Agricultural R&D Investment --
['Algeria' 'Antigua and Barbuda' 'Argentina' 'Australia' 'Azerbaijan'
 'Bangladesh' 'Barbados' 'Belarus' 'Belgium' 'Belize' 'Benin'
 'Bolivia (Plurinational State of)' 'Botswana' 'Brazil' 'Bulgaria'
 'Burkina Faso' 'Burundi' 'Cabo Verde' 'Cambodia' 'Cameroon'
 'Central African Republic' 'Chad' 'Chile' 'China' 'China, mainland'
 'Colombia' 'Congo' 'Costa Rica' "Côte d'Ivoire" 'Croatia' 'Cyprus'
 'Czechia' 'Democratic Republic of the Congo' 'Denmark' 'Dominica'
 'Dominican Republic' 'Ecuador' 'Egypt' 'El Salvador' 'Eritrea' 'Estonia'
 'Eswatini' 'Ethiopia' 'Finland' 'France' 'Gabon' 'Gambia' 'Ghana'
 'Greece' 'Grenada' 'Guatemala' 'Guinea' 'Guinea-Bissau' 'Honduras'
 'Hungary' 'Iceland' 'India' 'Indonesia' 'Iraq' 'Jamaica' 'Jordan' 'Kenya'
 "Lao People's Democratic Republic" 'Lebanon' 'Lesotho' 'Liberia'
 'Lithuania' 'Madagascar' 'Malawi' 'Malaysia' 'Mali' 'Mauritania'
 'Mauritius' 'Mexico' 'Morocco' 'Mozambique' 'Myanmar' 'Namibia' 'Nepal'
 'Nicaragua'

In [5]:
non_countries = {
    "World",
    "Arab World",
    "European Union",
    "Euro area",
    "OECD members",
    "High income",
    "Low income",
    "Lower middle income",
    "Upper middle income",
    "Middle income",
    "Low & middle income",

    "North America",
    "South Asia",
    "East Asia & Pacific",
    "Europe & Central Asia",
    "Latin America & Caribbean",
    "Sub-Saharan Africa",

    "Africa Eastern and Southern",
    "Africa Western and Central",
    "Central Europe and the Baltics",

    "Caribbean small states",
    "Pacific island small states",
    "Other small states",
    "Small states",

    "East Asia & Pacific (excluding high income)",
    "Europe & Central Asia (excluding high income)",
    "Latin America & Caribbean (excluding high income)",
    "Sub-Saharan Africa (excluding high income)",

    "East Asia & Pacific (IDA & IBRD countries)",
    "Europe & Central Asia (IDA & IBRD countries)",
    "Latin America & the Caribbean (IDA & IBRD countries)",
    "Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",
    "South Asia (IDA & IBRD)",
    "Sub-Saharan Africa (IDA & IBRD countries)",

    "Middle East, North Africa, Afghanistan & Pakistan",
    "Middle East, North Africa, Afghanistan & Pakistan (excluding high income)",

    "IBRD only",
    "IDA only",
    "IDA blend",
    "IDA total",
    "IDA & IBRD total",

    "Least developed countries: UN classification",
    "Fragile and conflict affected situations",
    "Heavily indebted poor countries (HIPC)",

    "Early-demographic dividend",
    "Late-demographic dividend",
    "Pre-demographic dividend",
    "Post-demographic dividend",

    "Not classified",
}

country_maps = {
    "Korea-": "Republic of Korea",
    "Korea, Rep.": "Republic of Korea",
    "Korea, Rep": "Republic of Korea",
    "Korea, Dem. People's Rep.": "Democratic People's Republic of Korea",
    "Moldova": "Republic of Moldova",
    "Tanzania": "United Republic of Tanzania",
    "United States": "United States of America",
    "United Kingdom": "United Kingdom of Great Britain and Northern Ireland",
    "Venezuela, RB": "Venezuela (Bolivarian Republic of)",
    "Venezuela, Rb": "Venezuela (Bolivarian Republic of)",
    "Bolivia": "Bolivia (Plurinational State of)",
    "Iran, Islamic Rep.": "Iran (Islamic Republic of)",
    "Egypt, Arab Rep.": "Egypt",
    "Yemen, Rep.": "Yemen",
    "Lao PDR": "Lao People's Democratic Republic",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Vietnam": "Viet Nam",
    "Turkiye": "Türkiye",
    "Turkey": "Türkiye",
    "Slovak Republic": "Slovakia",
    "Syrian Arab Republic": "Syria",
    "Somalia, Fed. Rep.": "Somalia",
    "Somalia": "Somalia",
    "Brunei Darussalam": "Brunei Darussalam",
    "Russian Federation": "Russian Federation",
    "Timor Leste": "Timor-Leste",
    "Trinidad And Tobago": "Trinidad and Tobago",
    "Guinea Bissau": "Guinea-Bissau",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.": "Congo",
    "Gambia, The": "Gambia",
    "Gambia,": "Gambia",
    "China, mainland": "China",
    "Bahamas, The": "Bahamas",
    "Bahamas,": "Bahamas",
    "Hong Kong SAR, China": "China, Hong Kong SAR",
    "Netherlands": "Netherlands (Kingdom of the)",
    "Eswatini": "Eswatini",
    "Czech Republic": "Czechia",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "St. Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "West Bank and Gaza": "Palestine",
    "Puerto Rico (US)": "Puerto Rico",
    "Micronesia, Fed. Sts.": "Micronesia",
    "Macao SAR, China": "China, Macao SAR",
}

In [6]:
def normalize_country(name):
    if pd.isna(name):
        return name
    name = str(name).strip()
    name = " ".join(name.split())
    name = unicodedata.normalize("NFKC", name)
    return name

def clean_country_column(df):
    df = df.copy()
    df["Country"] = df["Country"].apply(normalize_country).replace(country_maps)
    df = df[~df["Country"].isin(non_countries)]
    return df

In [7]:
RD = clean_country_column(RD)
PROD = clean_country_column(PROD)
GDP = clean_country_column(GDP)
POP = clean_country_column(POP)
POP_RUR = clean_country_column(POP_RUR)
RUR_LAND = clean_country_column(RUR_LAND)
TEMP = clean_country_column(TEMP)

In [8]:
common_countries = set(RD["Country"]).intersection(set(PROD["Country"]))

datasets = [RD, PROD, GDP, POP, POP_RUR, RUR_LAND, TEMP]
datasets = [df[df["Country"].isin(common_countries)].copy() for df in datasets]

RD, PROD, GDP, POP, POP_RUR, RUR_LAND, TEMP = datasets

print("Common countries:", len(common_countries))

Common countries: 110


#### X = Agricultural R&D Investment

In [9]:
RD.head()

,Country,Indicator,Year,Unit,Value
0,Algeria,Total Expenditure in Agricultural R&D (US$ PPP...,2009,million USD,71.2560
1,Algeria,Total Expenditure in Agricultural R&D (US$ PPP...,2010,million USD,66.0857
2,Algeria,Total Expenditure in Agricultural R&D (US$ PPP...,2011,million USD,77.6880
3,Algeria,Total Expenditure in Agricultural R&D (US$ PPP...,2012,million USD,84.0305
4,Antigua and Barbuda,Total Expenditure in Agricultural R&D (US$ PPP...,2007,million USD,1.3190


In [10]:
X = RD[["Country", "Year", "Value"]].rename(columns={"Year":"year", "Value":"rd"})
X.head()

,Country,year,rd
0,Algeria,2009,71.2560
1,Algeria,2010,66.0857
2,Algeria,2011,77.6880
3,Algeria,2012,84.0305
4,Antigua and Barbuda,2007,1.3190


#### Y = Value of Agricultural Production

In [11]:
PROD.head()

,Country,Element,Item,Year,Unit,Value
18,Algeria,Gross Production Value (constant 2014-2016 tho...,Agriculture,2005,1000 USD,13545693
19,Algeria,Gross Production Value (constant 2014-2016 tho...,Agriculture,2006,1000 USD,14221816
20,Algeria,Gross Production Value (constant 2014-2016 tho...,Agriculture,2007,1000 USD,13509876
21,Algeria,Gross Production Value (constant 2014-2016 tho...,Agriculture,2008,1000 USD,13571067
22,Algeria,Gross Production Value (constant 2014-2016 tho...,Agriculture,2009,1000 USD,16448620


In [12]:
Y = PROD[["Country", "Year", "Value"]].rename(columns={
    "Year": "year",
    "Value": "production"
})

Y = Y.groupby(["Country", "year"], as_index=False)["production"].sum()
Y.head()

,Country,year,production
0,Algeria,2005,13545693
1,Algeria,2006,14221816
2,Algeria,2007,13509876
3,Algeria,2008,13571067
4,Algeria,2009,16448620


#### GDP per capita (current US$)

In [13]:
GDP.head()

,Country,Indicator Name,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
9,Argentina,GDP per capita (current US$),5067.653423,5869.380290,7185.251551,8944.110266,8150.235270,10260.131310,12704.283180,12949.717490,12963.675770,12233.144410,13679.626500,12699.962310,14532.500930,11752.799890,9955.974787,8535.599380,10738.017920,13962.189410
12,Antigua and Barbuda,GDP per capita (current US$),14368.570880,16173.813630,18204.512400,18786.674310,16472.247850,15217.395080,14911.727510,15639.807810,15051.511360,15532.106210,16077.634080,16556.855180,16965.728830,18273.427070,18896.372180,15369.608230,17348.600700,20105.198910
13,Australia,GDP per capita (current US$),34535.207270,36658.824350,41128.247660,49811.206720,42954.849490,52313.893900,62799.389500,68301.440250,68476.678850,62817.206180,56970.361300,50084.444060,54117.547720,57410.173730,55194.758510,51983.452370,60758.904440,65169.519110
15,Azerbaijan,GDP per capita (current US$),1578.367330,2472.957257,3851.330152,5574.468104,4950.399490,5843.533768,7189.818692,7495.865277,7875.345367,7890.840281,5500.503646,3880.685228,4147.198142,4739.794312,4805.753718,4229.910649,5408.045352,7770.594223
16,Burundi,GDP per capita (current US$),147.234619,161.892756,166.162745,189.502075,198.431524,216.727705,230.069761,231.098788,234.844770,250.544492,254.402585,232.937812,246.060726,245.661678,234.310616,255.832454,264.980379,302.992505


In [14]:
gdp = GDP.melt(
    id_vars=["Country", "Indicator Name"],
    var_name="year",
    value_name="gdp_per_capita"
)

gdp["year"] = gdp["year"].astype(int)
gdp = gdp.drop(columns=["Indicator Name"])
gdp = gdp.groupby(["Country", "year"], as_index=False).mean()
gdp.head()

,Country,year,gdp_per_capita
0,Algeria,2005,3233.133396
1,Algeria,2006,3660.661047
2,Algeria,2007,4167.451699
3,Algeria,2008,5180.918815
4,Algeria,2009,4235.429905


**Population Total**

In [15]:
POP.head()

,Country,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
9,Argentina,39216789.0,39622115.0,40016763.0,40424148.0,40854831.0,41288694.0,41730660.0,42161721.0,42582455.0,43024071.0,43477012.0,43900313.0,44288894.0,44654882.0,44973465.0,45191965.0,45312281.0,45407904.0
12,Antigua and Barbuda,79611.0,80604.0,81704.0,82912.0,84173.0,85320.0,86332.0,87260.0,88064.0,88765.0,89409.0,89969.0,90468.0,90926.0,91364.0,91846.0,92349.0,92840.0
13,Australia,20176844.0,20450966.0,20827622.0,21249199.0,21691653.0,22031750.0,22340024.0,22733465.0,23128129.0,23475686.0,23815995.0,24190907.0,24592588.0,24963258.0,25334826.0,25649248.0,25685412.0,26018721.0
15,Azerbaijan,8391850.0,8484550.0,8581300.0,8763400.0,8947243.0,9054332.0,9173082.0,9295784.0,9416801.0,9535079.0,9649341.0,9757812.0,9854033.0,9939771.0,10024283.0,10093121.0,10137750.0,10141756.0
16,Burundi,7587299.0,7865547.0,8161874.0,8505637.0,8947646.0,9376444.0,9717978.0,10071028.0,10439341.0,10799785.0,11047580.0,11239451.0,11506762.0,11859446.0,12255336.0,12617036.0,12965481.0,13321097.0


In [16]:
pop = POP.melt(
    id_vars=["Country"],
    var_name="year",
    value_name="population"
)
pop["year"] = pop["year"].astype(int)
pop = pop.groupby(["Country", "year"], as_index=False).mean()
pop.head()

,Country,year,population
0,Algeria,2005,33109249.0
1,Algeria,2006,33623506.0
2,Algeria,2007,34189416.0
3,Algeria,2008,34816961.0
4,Algeria,2009,35490445.0


#### Rural population (% of total population)

In [17]:
POP_RUR.head()

,Country,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
9,Argentina,9.876626,9.696627,9.520337,9.347809,9.179098,9.006995,8.892771,8.795878,8.698549,8.602465,8.507698,8.414318,8.322397,8.232006,8.143216,8.056097,7.970722,7.887162
12,Antigua and Barbuda,70.826200,71.564742,72.279873,72.941409,73.519165,73.982955,74.305310,74.544438,74.752433,74.910027,75.000000,75.076019,75.149974,75.221857,75.291662,75.359384,75.425016,75.488552
13,Australia,14.380913,14.250746,14.103148,13.974618,13.872186,13.780140,13.673105,13.554874,13.425948,13.277871,13.122136,12.983370,12.871907,12.773232,12.698035,12.728653,12.764659,12.665673
15,Azerbaijan,47.497147,47.345985,47.171844,46.936695,46.864187,46.715163,46.491454,46.192356,45.817891,45.368082,44.842951,44.242520,43.566811,42.815847,41.869181,41.706531,41.634003,41.561718
16,Burundi,90.819383,90.535371,90.227108,89.893565,89.502010,89.015690,88.438335,87.775540,87.032904,86.216022,85.330491,84.381909,83.375873,82.317978,81.213822,80.069002,78.889114,77.679755


In [18]:
rural_pop = POP_RUR.melt(
    id_vars=["Country"],
    var_name="year",
    value_name="rural_pop_pct"
)
rural_pop["year"] = rural_pop["year"].astype(int)
rural_pop = rural_pop.groupby(["Country", "year"], as_index=False).mean()
rural_pop.head()

,Country,year,rural_pop_pct
0,Algeria,2005,36.340146
1,Algeria,2006,35.584565
2,Algeria,2007,34.830924
3,Algeria,2008,34.098514
4,Algeria,2009,33.490262


#### Agricultural land (% of land area)

In [19]:
RUR_LAND.head()

,Country,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
9,Argentina,47.473737,47.553395,47.491093,46.423563,45.430209,46.138912,45.791960,45.581304,44.921931,44.195918,44.123936,43.432424,42.982576,42.361393,42.605849,43.000486,43.109742,43.427644
12,Antigua and Barbuda,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545,20.454545
13,Australia,53.399373,53.278575,51.853677,50.673366,50.018484,48.992099,51.562813,50.385431,48.393710,48.738399,45.314424,44.539926,48.340618,46.658095,47.123772,46.252480,47.259237,47.259237
15,Azerbaijan,57.568352,57.552912,57.564535,57.568349,57.579095,57.670345,57.691935,57.687096,57.704545,57.700543,57.701753,57.735575,57.795601,57.825393,57.827812,57.838250,57.841500,57.840290
16,Burundi,70.833333,70.950156,71.066978,71.183801,71.300623,71.378505,71.690031,72.429907,73.481308,74.532710,75.584112,76.635514,77.686916,78.738318,79.789720,80.841121,81.892523,82.807963


In [20]:
rural_land = RUR_LAND.melt(
    id_vars=["Country"],
    var_name="year",
    value_name="rural_land_pct"
)
rural_land["year"] = rural_land["year"].astype(int)
rural_land = rural_land.groupby(["Country", "year"], as_index=False).mean()
rural_land.head()

,Country,year,rural_land_pct
0,Algeria,2005,17.302896
1,Algeria,2006,17.290300
2,Algeria,2007,17.320111
3,Algeria,2008,17.344043
4,Algeria,2009,17.373853


#### Average Annual Temperature

In [21]:
TEMP.head()

,Country,Year,Annual Mean,5-yr smooth
0,Dominican Republic,2005,24.41,24.42
1,Dominican Republic,2006,24.57,24.37
2,Dominican Republic,2007,24.33,24.31
3,Dominican Republic,2008,23.95,24.26
4,Dominican Republic,2009,24.31,24.24


In [22]:
temp = TEMP.rename(columns={"Year":"year","Annual Mean":"temperature"})
temp = temp[["Country","year","temperature"]]
temp = temp.groupby(["Country", "year"], as_index=False).mean()
temp.head()

,Country,year,temperature
0,Algeria,2005,23.58
1,Algeria,2006,23.74
2,Algeria,2007,23.58
3,Algeria,2008,23.43
4,Algeria,2009,23.87


### Merge Dataset

In [23]:
def force_unique(df, value_cols):
    return df.groupby(["Country", "year"], as_index=False)[value_cols].median()

X = force_unique(X, ["rd"])
Y = force_unique(Y, ["production"])
gdp = force_unique(gdp, ["gdp_per_capita"])
pop = force_unique(pop, ["population"])
rural_pop = force_unique(rural_pop, ["rural_pop_pct"])
rural_land = force_unique(rural_land, ["rural_land_pct"])
temp = force_unique(temp, ["temperature"])

In [24]:
datasets = {
    "X": X, "Y": Y,
    "gdp": gdp,
    "pop": pop,
    "rural_pop": rural_pop,
    "rural_land": rural_land,
    "temp": temp
}

for name, d in datasets.items():
    dup = d.duplicated(["Country","year"]).sum()
    print(name, "duplicates:", dup)

X duplicates: 0
Y duplicates: 0
gdp duplicates: 0
pop duplicates: 0
rural_pop duplicates: 0
rural_land duplicates: 0
temp duplicates: 0


In [25]:
df = X.copy()

for d in [Y, gdp, pop, rural_pop, rural_land, temp]:
    df = df.merge(d, on=["Country","year"], how="left")

df = df.sort_values(["Country","year"])

In [26]:
print("Duplicates:", df.duplicated(["Country","year"]).sum())

Duplicates: 0


### Fill Missing Values

In [27]:
print("\nMissing rates:")
print(df.isna().mean().sort_values(ascending=False).head(10))


Missing rates:
temperature       0.123148
Country           0.000000
year              0.000000
production        0.000000
rd                0.000000
gdp_per_capita    0.000000
population        0.000000
rural_pop_pct     0.000000
rural_land_pct    0.000000
dtype: float64


In [28]:
# 1. interpolate 
df["temperature"] = df.groupby("Country")["temperature"].transform(
    lambda x: x.interpolate(limit_direction="both")
)

df["temperature"] = df["temperature"].fillna(
    df.groupby("Country")["temperature"].transform("median")
)

# 3. global fallback  (для остатка)
df["temperature"] = df["temperature"].fillna(df["temperature"].median())

print(df.isna().mean().sort_values(ascending=False).head(10))

Country           0.0
year              0.0
rd                0.0
production        0.0
gdp_per_capita    0.0
population        0.0
rural_pop_pct     0.0
rural_land_pct    0.0
temperature       0.0
dtype: float64


### Feature Engineering

In [29]:
df["log_population"] = np.log1p(df["population"])
df["log_production"] = np.log1p(df["production"])
df["log_rd"] = np.log1p(df["rd"])

### Save Merged Dataset

In [30]:
df.to_csv(DATA_DIR / "merged_dataset.csv", index=False)